# 4. Foundation models: download the weights and extract frozen embeddings

Three publicly available ECG foundation models are used **with their published weights and without fine-tuning** (Supplementary Table 1). None was trained with race labels or on EchoNext; ECG-FM and HuBERT-ECG, but not ECGFounder, included MIMIC-IV-ECG in pretraining.

| Model | Source (pinned revision) | Licence of the weights |
|---|---|---|
| ECGFounder (Net1D, supervised on 150 diagnostic labels) | Hugging Face `PKUDigitalHealth/ECGFounder` @ `d9b17939`; architecture `net1d.py` from GitHub `PKUDigitalHealth/ECGFounder`, branch `main` @ `68d25f25` | MIT |
| ECG-FM (wav2vec 2.0-type transformer, self-supervised) | Hugging Face `wanglab/ecg-fm` @ `584219ea` (`mimic_iv_ecg_physionet_pretrained.pt`) | MIT |
| HuBERT-ECG, large version (HuBERT, self-supervised) | Hugging Face `Edoardo-Coppola/hubert-ecg-large` @ `f297337d` | CC BY-NC 4.0 |

The weights are **not** part of this repository. Section 4.1 downloads them (about 2.2 GB) into `MODEL_DIR` and checks their SHA-256 hashes. The comparators are the same three architectures with random weights (seed 0) and the same preprocessing.

**One embedding per ECG.** Each model receives its published preprocessing; MIMIC-IV ECGs are first converted to the format of the EchoNext release (notebook 03, section 3.6), so that both cohorts pass through identical pipelines.

* **ECGFounder**: linear interpolation to 500 Hz; one z-score over the whole 12 × 5000 array; no filtering. EchoNext leads are standardised one by one, which removes the amplitude differences between leads that ECGFounder expects, so each lead is first multiplied by a typical lead standard deviation (mV; the constants of fairseq-signals' preprocessing script). Embedding: the 1,024-dimensional output of the last stage, averaged over time.
* **ECG-FM**: linear interpolation to 500 Hz; z-score of each lead over the 10 s; two 5-second segments. Embedding: time average of the final encoder output (768 dimensions), averaged over the two segments. The network is re-implemented in plain PyTorch (the state-dict names of fairseq-signals load strictly).
* **HuBERT-ECG**: band-pass FIR filter 0.05–47 Hz at the native rate (the filter of biosppy, built with SciPy), FFT resampling to 500 Hz, min-max scaling of each lead to [−1, 1], two 5-second windows, each flattened (12 × 2500) and decimated by 5 as in the model's pretraining data loader. Embedding: time average of the last hidden state (960 dimensions), averaged over the two windows.

Extraction runs on the GPU (estimated from a timed sample on one RTX 5090: about 25 minutes for EchoNext and under an hour for MIMIC-IV) and is skipped for files that already exist.

In [ ]:
%run ./00_settings.ipynb
import collections
import hashlib
import math
import pickle
import sys
import urllib.request
import zipfile

import torch.nn as nn
from numpy.lib.format import open_memmap
from scipy import signal as ss
from scipy.interpolate import interp1d
from scipy.ndimage import median_filter

## 4.1 Download the weights (pinned revisions, SHA-256 checked)

The files go to `MODEL_DIR/<model>/`; a file that is already there with the right hash is not downloaded again. If the download stops with a certificate error (a network that inspects encrypted traffic), set the environment variable `REQUESTS_CA_BUNDLE` to a certificate bundle that includes your organisation's root certificate, or download the files by hand from the pages in the table above into those folders (the hashes are checked either way).

In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS = [  # (folder, Hugging Face repository, revision, file, SHA-256)
    ("ECGFounder", "PKUDigitalHealth/ECGFounder", "d9b1793951b2342f5f7e84f1ac03cd37f8a08724", "12_lead_ECGFounder.pth",
     "ee199f3781f4ae1f732973267f003da0a759ea12bddb0dd28a77faa60aca7997"),
    ("ECG-FM", "wanglab/ecg-fm", "584219ea492cdeef2e19ffbdf9c6ecc874ba427e", "mimic_iv_ecg_physionet_pretrained.pt",
     "4d0142bcb485eb9f0c7845e0c19ff3463f6ae9d0e458eab69136efe90ceb9b7e"),
    ("HuBERT-ECG-large", "Edoardo-Coppola/hubert-ecg-large", "f297337d4e33198800441e3d64e0dd393774637f", "model.safetensors",
     "9a0eb8484f73793aa678936984243da92ae80d7b81e2fe77d1ea6dee2bdf5ec2"),
    ("HuBERT-ECG-large", "Edoardo-Coppola/hubert-ecg-large", "f297337d4e33198800441e3d64e0dd393774637f", "config.json",
     "6b28e6a2321cf0ec341bb8e51b8d630d8de21bce0af218ff27fd99c12f2cbf6b"),
]
NET1D = ("https://raw.githubusercontent.com/PKUDigitalHealth/ECGFounder/68d25f25e323a4a423b9d9e8ea2e0af3f234bf22/net1d.py",   # head of the branch "main"
         "70800ec9870f38d072c22ffa6ab59aca699164d43a992e28e02064f6b4c32b72")


def sha256(path, bufsize=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(bufsize):
            h.update(chunk)
    return h.hexdigest()


for folder, repo, rev, fname, digest in WEIGHTS:
    dest = MODEL_DIR / folder
    p = dest / fname
    if not (p.exists() and sha256(p) == digest):
        p = Path(hf_hub_download(repo, fname, revision=rev, local_dir=dest))
    assert sha256(p) == digest, f"SHA-256 mismatch for {repo}/{fname}"
    print(f"{folder}/{fname}: {p.stat().st_size / 2**20:,.0f} MB, SHA-256 ok")
p = MODEL_DIR / "ECGFounder" / "net1d.py"
if not (p.exists() and sha256(p) == NET1D[1]):
    with urllib.request.urlopen(urllib.request.Request(NET1D[0], headers={"User-Agent": "ecgfm-race-encoding"}), timeout=60) as r:
        p.write_bytes(r.read())
assert sha256(p) == NET1D[1], "SHA-256 mismatch for net1d.py"
print("ECGFounder/net1d.py (official architecture code): SHA-256 ok")

## 4.2 Reading the checkpoints

The ECGFounder and ECG-FM checkpoints are PyTorch ZIP files that also contain configuration objects of other libraries. This loader reads **only the tensors**: no pickled code is executed and no third-party module is imported while unpickling (every other object becomes an inert placeholder).

In [ ]:
_DTYPES = {"FloatStorage": torch.float32, "DoubleStorage": torch.float64, "HalfStorage": torch.float16, "BFloat16Storage": torch.bfloat16,
           "LongStorage": torch.int64, "IntStorage": torch.int32, "ShortStorage": torch.int16, "CharStorage": torch.int8, "ByteStorage": torch.uint8,
           "BoolStorage": torch.bool}


class _Stub:
    def __init__(self, *a, **k): pass
    def __setstate__(self, s): self.state = s
    def __setitem__(self, k, v): pass
    def append(self, v): pass
    def extend(self, vs): pass


class _StorageType:
    def __init__(self, name):
        self.dtype = _DTYPES[name]


class _LazyStorage:
    def __init__(self, stype, key, numel):
        self.stype, self.key = stype, key


class _LazyTensor:
    def __init__(self, storage, offset, size, stride, *rest):
        self.storage, self.offset, self.size, self.stride = storage, offset, tuple(size), tuple(stride)


class _Unpickler(pickle.Unpickler):
    _classes = {}

    def find_class(self, module, name):
        if f"{module}.{name}" == "collections.OrderedDict":
            return collections.OrderedDict
        if name == "_rebuild_tensor_v2":
            return _LazyTensor
        if name in ("_rebuild_parameter", "_rebuild_parameter_with_state"):
            return lambda data, *a: data
        if name == "_rebuild_from_type_v2":
            return lambda func, new_type, args, state: func(*args)
        if module == "torch" and name in _DTYPES:
            return _StorageType(name)
        return self._classes.setdefault(f"{module}.{name}", type(name, (_Stub,), {}))

    def persistent_load(self, pid):                     # ('storage', storage_type, key, location, numel)
        return _LazyStorage(pid[1], pid[2], pid[4])


class Checkpoint:
    def __init__(self, path):
        self.zf = zipfile.ZipFile(path)
        pkl = [n for n in self.zf.namelist() if n.endswith("/data.pkl")][0]
        self.prefix = pkl.rsplit("/", 1)[0]
        with self.zf.open(pkl) as f:
            self.obj = _Unpickler(f).load()

    def tensor(self, lt):
        buf = bytearray(self.zf.read(f"{self.prefix}/data/{lt.storage.key}"))
        flat = torch.frombuffer(buf, dtype=lt.storage.stype.dtype) if len(buf) else torch.empty(0, dtype=lt.storage.stype.dtype)
        return flat.as_strided(lt.size, lt.stride, lt.offset).clone()

    def state_dict(self, key=None):
        sd = self.obj if key is None else self.obj[key]
        return collections.OrderedDict((k, self.tensor(v)) for k, v in sd.items() if isinstance(v, _LazyTensor))

## 4.3 The three models and their preprocessing

Input of every embedder: float32 array B × 12 × 2500 (250 Hz, leads I, II, III, aVR, aVL, aVF, V1–V6), standardised as in the EchoNext release.

In [ ]:
FS_IN = 250
# per-lead population SD (mV; leads I..V6) of fairseq-signals' scripts/preprocess/ecg/preprocess.py ("STDs"): restores typical lead amplitude ratios for ECGFounder
REF_STD_MV = np.array([0.12781512, 0.14941794, 0.12525026, 0.12572662, 0.10236586, 0.12168122,
                       0.1870286, 0.28818728, 0.29218111, 0.29028273, 0.26973001, 0.23605948], np.float32)


def linear_resample(x, fs_in, fs_out):
    """fairseq-signals `resample`: linear interpolation onto an np.linspace grid, along the last axis."""
    n = x.shape[-1]
    m = int(n * (fs_out / fs_in))
    pos = np.linspace(0, m - 1, n)
    return interp1d(pos, x, kind="linear", axis=-1)(np.arange(m)).astype(np.float32)


def run_batches(fn, x, bs, dev):
    """Apply a model function to x in batches on the GPU; returns float32 numpy."""
    outs = []
    for i in range(0, len(x), bs):
        outs.append(fn(torch.from_numpy(np.ascontiguousarray(x[i:i + bs])).to(dev)).float().cpu().numpy())
    return np.concatenate(outs, 0)


# ---------------------------------------------------------------------------- ECGFounder
class ECGFounderEmbedder:
    dim = 1024

    def __init__(self, dev, pretrained=True, seed=0):
        sys.path.insert(0, str(MODEL_DIR / "ECGFounder"))
        from net1d import Net1D                          # official architecture code (downloaded in 4.1)
        torch.manual_seed(seed)
        self.model = Net1D(in_channels=12, base_filters=64, ratio=1, filter_list=[64, 160, 160, 400, 400, 1024, 1024],
                           m_blocks_list=[2, 2, 2, 3, 3, 4, 4], kernel_size=16, stride=2, groups_width=16, verbose=False, use_bn=False,
                           use_do=False, n_classes=150, return_features=True)
        self.load_report = f"random weights (seed {seed})"
        if pretrained:
            sd = Checkpoint(MODEL_DIR / "ECGFounder" / "12_lead_ECGFounder.pth").state_dict("state_dict")
            self.model.load_state_dict(sd, strict=True)
            self.load_report = f"strict load ({len(sd)} tensors)"
        self.model.eval().to(dev)
        self.dev = dev

    @staticmethod
    def preprocess(x):
        x = x * REF_STD_MV[None, :, None]                   # restore lead amplitude ratios
        x = linear_resample(x, FS_IN, 500)                  # B x 12 x 5000
        mu, sd = x.mean(axis=(1, 2), keepdims=True), x.std(axis=(1, 2), keepdims=True)
        return ((x - mu) / (sd + 1e-8)).astype(np.float32)

    @torch.inference_mode()
    def embed_prepared(self, xp, bs=64):
        return np.concatenate([self.model(torch.from_numpy(xp[i:i + bs]).to(self.dev))[1].float().cpu().numpy() for i in range(0, len(xp), bs)])


# ---------------------------------------------------------------------------- ECG-FM (fairseq-signals ECGTransformerModel, features only)
class _SamePad(nn.Module):
    def __init__(self, k):
        super().__init__()
        self.remove = 1 if k % 2 == 0 else 0

    def forward(self, x):
        return x[:, :, :-self.remove] if self.remove else x


class _Fp32GroupNorm(nn.GroupNorm):
    def forward(self, x):
        return TF.group_norm(x.float(), self.num_groups, self.weight.float(), self.bias.float(), self.eps).type_as(x)


class _MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(d, d) for _ in range(4))

    def forward(self, x):                                   # B x T x C
        Bn, T, Cn = x.shape
        q, k, v = (p(x).view(Bn, T, self.h, Cn // self.h).transpose(1, 2) for p in (self.q_proj, self.k_proj, self.v_proj))
        o = TF.scaled_dot_product_attention(q, k, v)        # scale = head_dim ** -0.5, as in fairseq
        return self.out_proj(o.transpose(1, 2).reshape(Bn, T, Cn))


class _Layer(nn.Module):                                    # transformer layer, post-layer-norm
    def __init__(self, d, ffn, h):
        super().__init__()
        self.self_attn = _MHA(d, h)
        self.self_attn_layer_norm = nn.LayerNorm(d)
        self.fc1, self.fc2 = nn.Linear(d, ffn), nn.Linear(ffn, d)
        self.final_layer_norm = nn.LayerNorm(d)

    def forward(self, x):
        x = self.self_attn_layer_norm(x + self.self_attn(x))
        y = self.fc2(TF.gelu(self.fc1(x).float()).type_as(x))
        return self.final_layer_norm(x + y)


class _Encoder(nn.Module):
    def __init__(self, d, ffn, h, n):
        super().__init__()
        self.layers = nn.ModuleList([_Layer(d, ffn, h) for _ in range(n)])
        self.layer_norm = nn.LayerNorm(d)

    def forward(self, x):
        x = self.layer_norm(x)                              # layer_norm_first = False: one layer norm before the layers
        for layer in self.layers:
            x = layer(x)
        return x


class _FeatureExtractor(nn.Module):                         # 4 x Conv1d(256, kernel 2, stride 2), no bias; group norm after the first
    def __init__(self, in_d=12, dim=256, n=4):
        super().__init__()
        blocks = []
        for i in range(n):
            conv = nn.Conv1d(in_d if i == 0 else dim, dim, 2, stride=2, bias=False)
            nn.init.kaiming_normal_(conv.weight)
            if i == 0:
                blocks.append(nn.Sequential(conv, nn.Dropout(0.0), _Fp32GroupNorm(dim, dim, affine=True), nn.GELU()))
            else:
                blocks.append(nn.Sequential(conv, nn.Dropout(0.0), nn.GELU()))
        self.conv_layers = nn.ModuleList(blocks)

    def forward(self, x):
        for b in self.conv_layers:
            x = b(x)
        return x


class _ConvPos(nn.Module):
    def __init__(self, d=768, k=128, groups=16):
        super().__init__()
        conv = nn.Conv1d(d, d, kernel_size=k, padding=k // 2, groups=groups)
        nn.init.normal_(conv.weight, mean=0, std=math.sqrt(4.0 / (k * d)))
        nn.init.constant_(conv.bias, 0)
        self.pos_conv = nn.Sequential(conv, _SamePad(k), nn.GELU())

    def forward(self, x):                                   # B x T x C
        return self.pos_conv(x.transpose(1, 2)).transpose(1, 2)


class ECGFMNet(nn.Module):
    def __init__(self, d=768, ffn=3072, h=12, n=12):      # the released checkpoint has 12 layers of width 768
        super().__init__()
        self.feature_extractor = _FeatureExtractor()
        self.layer_norm = nn.LayerNorm(256)
        self.post_extract_proj = nn.Linear(256, d)
        self.conv_pos = _ConvPos(d)
        self.encoder = _Encoder(d, ffn, h, n)
        for m in self.encoder.modules():                    # fairseq init_bert_params on the transformer encoder
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, src):                                 # B x 12 x T (500 Hz)
        f = self.feature_extractor(src).transpose(1, 2)
        f = self.post_extract_proj(self.layer_norm(f))
        x = f + self.conv_pos(f)
        return self.encoder(x)                              # B x T' x 768

    def load_fairseq(self, sd):
        g = sd.pop("conv_pos.pos_conv.0.weight_g")
        v = sd.pop("conv_pos.pos_conv.0.weight_v")
        sd["conv_pos.pos_conv.0.weight"] = torch._weight_norm(v, g, 2)          # weight normalisation folded into the weight
        dropped = sorted({k.split(".")[0] for k in sd if k.split(".")[0] in ("mask_emb", "quantizer", "project_q", "final_proj")})
        sd = {k: v for k, v in sd.items() if k.split(".")[0] not in dropped}   # pretraining-only modules
        self.load_state_dict(sd, strict=True)
        return f"strict load ({len(sd)} tensors; pretraining-only modules dropped: {dropped})"


class ECGFMEmbedder:
    dim = 768

    def __init__(self, dev, pretrained=True, seed=0):
        torch.manual_seed(seed)
        self.model = ECGFMNet()
        self.load_report = f"random weights (seed {seed})"
        if pretrained:
            self.load_report = self.model.load_fairseq(Checkpoint(MODEL_DIR / "ECG-FM" / "mimic_iv_ecg_physionet_pretrained.pt").state_dict("model"))
        self.model.eval().to(dev)
        self.dev = dev

    @staticmethod
    def preprocess(x):
        x = linear_resample(x, FS_IN, 500)                  # B x 12 x 5000
        x = x - x.mean(-1, keepdims=True)
        sd = x.std(-1, keepdims=True)
        x = np.where(sd == 0, 0.0, x / np.where(sd == 0, 1.0, sd)).astype(np.float32)
        return [x[..., :2500], x[..., 2500:5000]]           # two 5-s segments

    @torch.inference_mode()
    def _segment(self, t):
        return self.model(t).mean(1)

    def embed_prepared(self, segs, bs=128):
        return np.mean([run_batches(self._segment, s, bs, self.dev) for s in segs], axis=0)


# ---------------------------------------------------------------------------- HuBERT-ECG (transformers HubertModel)
def hubert_config(path):
    from transformers import HubertConfig
    cfg = json.loads(Path(path).read_text())
    for k in ["model_type", "architectures", "auto_map", "ensemble_length", "vocab_sizes", "id2label", "label2id", "transformers_version",
              "torch_dtype", "dtype", "task_type", "use_label_embedding", "classifier_hidden_size", "classifier_dropout_prob", "activation"]:
        cfg.pop(k, None)
    return HubertConfig(**cfg)


# band-pass FIR 0.05-47 Hz of order int(0.3 * fs), as biosppy.signals.tools.get_filter(ftype="FIR", band="bandpass", ...) builds it
_n_taps = int(0.3 * FS_IN) + (1 - int(0.3 * FS_IN) % 2)     # odd number of taps (75 at 250 Hz)
FIR_B = ss.firwin(numtaps=_n_taps, cutoff=2.0 * np.array([0.05, 47]) / FS_IN, pass_zero=False)


class HuBERTECGEmbedder:
    def __init__(self, dev, pretrained=True, seed=0):
        from safetensors.torch import load_file
        from transformers import HubertModel
        d = MODEL_DIR / "HuBERT-ECG-large"
        torch.manual_seed(seed)
        self.config = hubert_config(d / "config.json")
        self.model = HubertModel(self.config)
        self.dim = self.config.hidden_size
        self.load_report = f"random weights (seed {seed})"
        if pretrained:
            sd = load_file(str(d / "model.safetensors"))
            sd = {k: v for k, v in sd.items() if k.split(".")[0] not in ("final_proj", "label_embedding")}   # pretraining-only modules
            missing, unexpected = self.model.load_state_dict(sd, strict=False)
            missing = [m for m in missing if m != "masked_spec_embed"]
            if missing or unexpected:
                raise RuntimeError(f"HuBERT-ECG: missing {missing[:5]}, unexpected {unexpected[:5]}")
            self.load_report = f"load ({len(sd)} tensors)"
        self.model.eval().to(dev)
        self.dev = dev

    @staticmethod
    def preprocess(x):
        x = ss.filtfilt(FIR_B, [1], x, axis=-1)              # band-pass at the native rate
        x = ss.resample(x, int(x.shape[-1] * 500 / FS_IN), axis=-1)       # FFT resampling to 500 Hz (5000 samples)
        mn, mx = x.min(-1, keepdims=True), x.max(-1, keepdims=True)
        x = 2 * (x - mn) / (mx - mn + 1e-8) - 1              # each lead to [-1, 1]
        wins = []
        for s in (0, 2500):
            w = x[..., s:s + 2500].reshape(len(x), -1)       # flatten 12 x 2500 (lead after lead)
            w = ss.decimate(w, 5, axis=-1)                   # -> 6000 samples, as the pretraining data loader
            wins.append(np.ascontiguousarray(w, dtype=np.float32))
        return wins

    @torch.inference_mode()
    def _window(self, t):
        return self.model(input_values=t).last_hidden_state.mean(1)

    def embed_prepared(self, wins, bs=128):
        return np.mean([run_batches(self._window, w, bs, self.dev) for w in wins], axis=0)


def build_encoders(dev):
    """The three pretrained models and the same architectures with random weights (seed 0)."""
    return {"ecgfounder": ECGFounderEmbedder(dev, pretrained=True), "ecgfounder_random": ECGFounderEmbedder(dev, pretrained=False, seed=0),
            "ecgfm": ECGFMEmbedder(dev, pretrained=True), "ecgfm_random": ECGFMEmbedder(dev, pretrained=False, seed=0),
            "hubert_large": HuBERTECGEmbedder(dev, pretrained=True), "hubert_large_random": HuBERTECGEmbedder(dev, pretrained=False, seed=0)}


def prepare(x):
    """Preprocessing once per model family (the pretrained and random-weight versions share it)."""
    return {"ecgfounder": ECGFounderEmbedder.preprocess(x), "ecgfm": ECGFMEmbedder.preprocess(x), "hubert_large": HuBERTECGEmbedder.preprocess(x)}


def embed_all(encoders, prepared):
    return {name: enc.embed_prepared(prepared[name.replace("_random", "")]) for name, enc in encoders.items()}

## 4.4 Load the models (Supplementary Table 1)

Parameter counts include every module used for the embeddings and exclude modules needed only for pretraining.

In [ ]:
dev = device()
if dev.type != "cuda":
    print("No GPU: extraction will be very slow.")
encoders = build_encoders(dev)
info = {name: {"dim": int(enc.dim), "parameters": int(sum(p.numel() for p in enc.model.parameters())), "load": enc.load_report}
        for name, enc in encoders.items()}
save_json(info, results("04_embeddings") / "models.json")
pd.DataFrame(info).T.assign(parameters_millions=lambda d: (d.parameters.astype(float) / 1e6).round(1))

## 4.5 EchoNext embeddings

One sequential pass over each split file (large chunks, so that a hard disk is read efficiently). Output: `WORK_DIR/embeddings/echonext/<model>/<split>.npy` (float32, rows in the order of the split file).

In [ ]:
CHUNK = 1024
root = work("embeddings", "echonext")
for split in ["val", "test", "train", "no_split"]:
    wv = echonext_waveforms(split)
    n = wv.shape[0]
    todo = [k for k in encoders if not (root / k / f"{split}.done.json").exists()]
    if not todo:
        print(f"[{split}] done before, skipped")
        continue
    outs = {}
    for k in todo:
        (root / k).mkdir(parents=True, exist_ok=True)
        outs[k] = open_memmap(root / k / f"{split}.npy", mode="w+", dtype=np.float32, shape=(n, encoders[k].dim))
    t0 = time.time()
    for s in range(0, n, CHUNK):
        x = np.asarray(wv[s:s + CHUNK], dtype=np.float32)
        x = np.ascontiguousarray(x[:, 0].transpose(0, 2, 1))                 # B x 12 x 2500
        res = embed_all({k: encoders[k] for k in todo}, prepare(x))
        for k in todo:
            outs[k][s:s + len(x)] = res[k]
    for k in todo:
        outs[k].flush()
        finite = bool(np.isfinite(np.asarray(outs[k][:min(n, 5000)])).all())
        save_json({"split": split, "n": n, "dim": encoders[k].dim, "finite_first5000": finite}, root / k / f"{split}.done.json")
    del outs
    print(f"[{split}] {n:,} ECGs in {(time.time() - t0) / 60:.1f} min")

## 4.6 MIMIC-IV embeddings

Every candidate ECG of notebook 03 is converted to the EchoNext format (250 Hz, median filter of length 3, per-lead clipping and standardisation with the constants of section 3.6) and then embedded exactly like an EchoNext ECG. The CPU preprocessing of the next chunks runs in background threads while the GPU embeds the current one. Output: `WORK_DIR/embeddings/mimic/<model>/all.npy` (float32, one row per candidate ECG; index with the cohort column `row`).

In [ ]:
from collections import deque
from concurrent.futures import ThreadPoolExecutor

fmt = load_json(RESULTS_DIR / "03_mimic" / "format_stats.json")
LO, HI, MU, SD = (np.asarray(fmt[k], np.float32)[None, :, None] for k in ("clip_lo_mv", "clip_hi_mv", "mean_mv", "sd_mv"))


def to_echonext_format(raw):
    """int16 (200 units per mV, 500 Hz) -> float32 B x 12 x 2500 in the format of the EchoNext release."""
    x = raw.astype(np.float32) / 200.0
    y = ss.resample_poly(x, 1, 2, axis=-1).astype(np.float32)                  # 500 -> 250 Hz, anti-aliased
    k = fmt["median_kernel"]
    y = median_filter(y, size=(1, 1, k), mode="nearest") if k > 1 else y
    return ((np.clip(y, LO, HI) - MU) / SD).astype(np.float32)


root = work("embeddings", "mimic")
if all((root / k / "all.done.json").exists() for k in encoders):
    print("MIMIC-IV embeddings done before, skipped")
else:
    Wv = np.load(WORK_DIR / "mimic" / "waveforms_int16.npy", mmap_mode="r")
    n = Wv.shape[0]
    outs = {}
    for k in encoders:
        (root / k).mkdir(parents=True, exist_ok=True)
        outs[k] = open_memmap(root / k / "all.npy", mode="w+", dtype=np.float32, shape=(n, encoders[k].dim))

    def job(s):                                                                  # disk read + CPU preprocessing of one chunk
        return s, prepare(to_echonext_format(np.asarray(Wv[s:min(s + CHUNK, n)])))

    t0 = time.time()
    starts = iter(range(0, n, CHUNK))
    with ThreadPoolExecutor(4) as ex:
        pending = deque(ex.submit(job, s) for s, _ in zip(starts, range(4)))
        i = 0
        while pending:
            s, prepared = pending.popleft().result()
            nxt = next(starts, None)
            if nxt is not None:
                pending.append(ex.submit(job, nxt))
            res = embed_all(encoders, prepared)
            for k in encoders:
                outs[k][s:s + len(res[k])] = res[k]
            i += 1
            if i % 20 == 0:
                done = s + len(res[k])
                print(f"{done:,}/{n:,} ECGs, {done / (time.time() - t0):.0f} per second", flush=True)
    for k in encoders:
        outs[k].flush()
        save_json({"n": n, "dim": encoders[k].dim, "finite_first5000": bool(np.isfinite(np.asarray(outs[k][:5000])).all())}, root / k / "all.done.json")
    del outs
    print(f"MIMIC-IV: {n:,} ECGs in {(time.time() - t0) / 60:.1f} min")
if torch.cuda.is_available():
    print(f"peak GPU memory reserved: {torch.cuda.max_memory_reserved() / 2**30:.1f} GB")